# Market Regime Filter — Protecting Momentum in Bear Markets

Momentum strategies ‘blow up’ in sustained drawdowns because winners stop winning when the entire market is falling.
A **regime filter** gates the strategy: only trade when the market is in a defined risk-on state; hold cash otherwise.

Two filters are compared:
1. **Trend filter** — long only when SPY closes above its 200-day SMA (classic risk-on rule).
2. **Volatility regime** — long only when SPY’s 20-day realised vol is below 20 % annualised.

In [1]:
import os, sys

_cwd = os.getcwd()
_root = _cwd if os.path.exists(os.path.join(_cwd, 'pyproject.toml')) \
    else os.path.abspath(os.path.join(_cwd, '..'))
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.makedirs('results', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from backtest_engine.data.loader import load_prices
from backtest_engine.strategies.momentum import momentum_signals
from backtest_engine.strategies.regime import (
    apply_regime_filter,
    trend_filter,
    volatility_regime,
)
from backtest_engine.backtest.engine import run_backtest
from backtest_engine.metrics.performance import (
    cagr, sharpe_ratio, max_drawdown, calmar_ratio, compare_strategies,
)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

SECTOR_ETFS = ['XLK', 'XLF', 'XLV', 'XLE', 'XLY',
               'XLP', 'XLI', 'XLB', 'XLU', 'XLRE', 'XLC']
START   = '2010-01-01'
END     = '2024-12-31'
CAPITAL = 10_000
TOP_N   = 3

## 1. Load Data

Same universe as notebook 02.  SPY is loaded separately as the regime filter signal.

In [2]:
prices = load_prices(SECTOR_ETFS, START, END, cache=True)

spy_raw = load_prices(['SPY'], START, END, cache=True)
common  = prices.index.intersection(spy_raw.index)
prices  = prices.loc[common]
spy     = spy_raw.loc[common, 'SPY']

print(f'Sector universe : {list(prices.columns)}')
print(f'Date range      : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Trading days    : {len(prices)}')
prices.tail(3)

Sector universe : ['XLK', 'XLF', 'XLV', 'XLE', 'XLY', 'XLP', 'XLI', 'XLB', 'XLU', 'XLRE', 'XLC']
Date range      : 2018-06-19 → 2024-12-30
Trading days    : 1644


,XLK,XLF,XLV,XLE,XLY,XLP,XLI,XLB,XLU,XLRE,XLC
Date,,,,,,,,,,,
2024-12-26,119.451431,48.188152,136.625366,40.652626,115.565979,77.136940,132.068970,41.680855,36.833008,39.245575,97.538261
2024-12-27,117.863312,47.834908,135.989197,40.647820,113.656364,76.759445,131.085251,41.456345,36.726807,38.928692,96.671600
2024-12-30,116.364517,47.373734,134.315643,40.643013,111.830856,75.859230,129.767136,40.899948,36.572330,38.746243,95.657219


## 2. Build Regime Filters

Both filters operate on SPY so they reflect broad market conditions, not sector-specific noise.

In [3]:
# 1. Trend filter: SPY > 200-day SMA
sma_regime = trend_filter(spy, sma_window=200)

# 2. Vol regime: SPY 20-day realised vol < 20 % annualised
spy_returns      = spy.pct_change().fillna(0.0)
vol_regime_mask  = volatility_regime(spy_returns, threshold=0.20, lookback=20)

sma_pct = sma_regime.mean() * 100
vol_pct = vol_regime_mask.mean() * 100
print(f'Trend filter     : {sma_pct:.1f} % of days risk-on')
print(f'Vol-regime filter: {vol_pct:.1f} % of days risk-on')

Trend filter     : 70.8 % of days risk-on
Vol-regime filter: 74.8 % of days risk-on


## 3. Generate Momentum Signals and Apply Filters

In [4]:
signals_base = momentum_signals(prices, lookback_months=12, skip_months=1, top_n=TOP_N)
signals_sma  = apply_regime_filter(signals_base, sma_regime)
signals_vol  = apply_regime_filter(signals_base, vol_regime_mask)

def active_days(sig):
    return int((sig.abs().sum(axis=1) > 0).sum())

print('Active days (rows where any position > 0):')
print(f'  Baseline   : {active_days(signals_base)} / {len(signals_base)}')
print(f'  SMA filter : {active_days(signals_sma)} / {len(signals_sma)}')
print(f'  Vol filter : {active_days(signals_vol)} / {len(signals_vol)}')

Active days (rows where any position > 0):
  Baseline   : 1385 / 1644
  SMA filter : 1106 / 1644
  Vol filter : 1043 / 1644


## 4. Backtest All Three Variants

In [5]:
common_kwargs = dict(
    initial_capital      = CAPITAL,
    transaction_cost_bps = 5.0,
    slippage_bps         = 2.0,
    sizing_method        = 'equal_weight',
)

result_base = run_backtest(prices, signals_base, **common_kwargs)
result_sma  = run_backtest(prices, signals_sma,  **common_kwargs)
result_vol  = run_backtest(prices, signals_vol,  **common_kwargs)

# SPY buy-and-hold: align to same dates
spy_df  = spy.to_frame('SPY')
spy_sig = pd.DataFrame({'SPY': 1.0}, index=prices.index)
result_spy = run_backtest(spy_df, spy_sig, **common_kwargs)

print('Backtests complete.')

Backtests complete.


## 5. Performance Comparison

In [6]:
compare_strategies({
    f'Momentum baseline (top {TOP_N})': result_base.returns,
    '+ SPY 200-SMA filter'            : result_sma.returns,
    '+ Vol-regime filter (vol < 20 %)': result_vol.returns,
    'SPY buy-and-hold'                : result_spy.returns,
})


Strategy Comparison
────────────────────────────────────────────────────────────────────────────────
  Metric                Momentum baseline (top 3)+ SPY 200-SMA filter+ Vol-regime filter (vol < 20 %)SPY buy-and-hold
────────────────────────────────────────────────────────────────────────────────
  Total Return                +122.95%       +78.81%       +58.03%      +136.93%
  CAGR                         +13.08%        +9.32%        +7.27%       +14.14%
  Annual Volatility             19.25%        13.07%        12.58%        19.64%
  Sharpe Ratio                   0.735         0.747         0.621         0.772
  Sortino Ratio                  0.971         0.997         0.804         1.006
  Max Drawdown                 -30.38%       -17.45%       -14.63%       -33.72%
  Calmar Ratio                   0.430         0.534         0.497         0.419
  Win Rate                      45.68%        36.56%        34.06%        55.17%
  Profit Factor                  1.161         1.16

In [7]:
def pct_in_market(sig):
    return float((sig.abs().sum(axis=1) > 0).mean() * 100)

print(f'  Baseline   : {pct_in_market(signals_base):.1f} % of days invested')
print(f'  SMA filter : {pct_in_market(signals_sma):.1f} % of days invested')
print(f'  Vol filter : {pct_in_market(signals_vol):.1f} % of days invested')

  Baseline   : 84.2 % of days invested
  SMA filter : 67.3 % of days invested
  Vol filter : 63.4 % of days invested


## 6. Equity Curves with Risk-Off Shading

Grey bands mark days when each filter was risk-off (holding cash).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=True)

variants = [
    (f'Baseline (top {TOP_N})', result_base, None,            '#1f77b4'),
    ('+ SPY 200-SMA filter',    result_sma,  sma_regime,      '#2ca02c'),
    ('+ Vol-regime filter',     result_vol,  vol_regime_mask, '#d62728'),
]
spy_eq = (1.0 + result_spy.returns).cumprod() * CAPITAL

for ax, (label, res, mask, color) in zip(axes, variants, strict=True):
    equity = (1.0 + res.returns).cumprod() * CAPITAL
    ax.plot(equity.index, equity, color=color, linewidth=1.5, label=label)
    ax.plot(spy_eq.index, spy_eq, color='black', linewidth=0.8,
            linestyle='--', alpha=0.4, label='SPY')

    if mask is not None:
        risk_off = (~mask.reindex(equity.index).fillna(False)).values
        in_band, band_start = False, None
        dates = equity.index
        for date, off in zip(dates, risk_off, strict=True):
            if off and not in_band:
                band_start, in_band = date, True
            elif not off and in_band:
                ax.axvspan(band_start, date, alpha=0.12, color='grey', lw=0)
                in_band = False
        if in_band:
            ax.axvspan(band_start, dates[-1], alpha=0.12, color='grey', lw=0)

    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'${x:,.0f}')
    )
    ax.set_title(label, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)

axes[-1].set_xlabel('Date')
fig.suptitle(f'Regime Filter Comparison — ${CAPITAL:,} initial capital',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/04_regime_equity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/04_regime_equity.png')

## 7. Verdict Numbers

In [9]:
for label, res, sig in [
    (f'Baseline (top {TOP_N})',           result_base, signals_base),
    ('+ SPY 200-SMA filter',              result_sma,  signals_sma),
    ('+ Vol-regime filter (vol < 20 %)',  result_vol,  signals_vol),
]:
    r = res.returns
    print(
        f'{label:<42} '
        f'CAGR {cagr(r)*100:+.2f}%  '
        f'Sharpe {sharpe_ratio(r):.3f}  '
        f'MaxDD {max_drawdown(r)*100:.1f}%  '
        f'Calmar {calmar_ratio(r):.2f}  '
        f'In-mkt {pct_in_market(sig):.1f}%'
    )

Baseline (top 3)                           CAGR +13.08%  Sharpe 0.735  MaxDD -30.4%  Calmar 0.43  In-mkt 84.2%
+ SPY 200-SMA filter                       CAGR +9.32%  Sharpe 0.747  MaxDD -17.4%  Calmar 0.53  In-mkt 67.3%
+ Vol-regime filter (vol < 20 %)           CAGR +7.27%  Sharpe 0.621  MaxDD -14.6%  Calmar 0.50  In-mkt 63.4%


## Honest Verdict

**Trend filter (SPY > 200-SMA):**
- CAGR: **+9.32 %** vs baseline **+13.08 %** — costs −3.76 pp of compounding
- Sharpe: **0.747** vs **0.735** — marginal improvement (+0.012)
- Max drawdown: **−17.4 %** vs **−30.4 %** — cuts worst loss by 13 pp
- Calmar: **0.53** vs **0.43** — meaningfully better return per unit of pain
- Days invested: **67.3 %** (down from 84.2 % baseline)

**Vol-regime filter (20-day vol < 20 %):**
- CAGR: **+7.27 %** vs baseline **+13.08 %** — costs −5.81 pp of compounding
- Sharpe: **0.621** vs **0.735** — actually *hurts* risk-adjusted return
- Max drawdown: **−14.6 %** vs **−30.4 %** — best drawdown of the three
- Calmar: **0.50** vs **0.43** — slightly better, but worse than SMA filter
- Days invested: **63.4 %** (down from 84.2 % baseline)

**What the numbers show:**

The **SMA filter wins** on a risk-adjusted basis.
It barely moves Sharpe (0.735 → 0.747) while cutting max drawdown almost in half
(−30.4 % → −17.4 %), lifting the Calmar ratio from 0.43 to 0.53.
The cost is real: −3.76 pp of annual compounding.

The **vol-regime filter is too aggressive** in this universe.
It exits before big drawdowns (best max DD: −14.6 %) but also misses the sharpest
recovery rallies. Sharpe falls from 0.735 to 0.621 — a clear warning that whipsawing
in and out of positions destroys more alpha than the drawdown protection gains back.

**The key insight: regime filters are *drawdown surgery*, not alpha generators.**
The SMA filter makes momentum *survivable* in bear markets without gutting the edge.
For a portfolio with a hard drawdown constraint (pension fund, risk-managed account),
trading −3.8 pp CAGR for −13 pp peak drawdown is likely worth it.
For a pure-alpha investor who can stomach −30 % drawdowns, the unfiltered baseline
compounts more capital over the full cycle.

| | Baseline | + SMA filter | + Vol filter | SPY |
|---|---|---|---|---|
| CAGR | +13.08 % | +9.32 % | +7.27 % | +14.14 % |
| Sharpe | 0.735 | **0.747** | 0.621 | 0.772 |
| Max DD | −30.4 % | −17.4 % | **−14.6 %** | −33.7 % |
| Calmar | 0.43 | **0.53** | 0.50 | 0.42 |
| In-market | 84.2 % | 67.3 % | 63.4 % | 100 % |